# Introduccion

Hola de nuevo, espero que no hayas tenido problemas con el modulo anteior de descarga :smile:, porque ahora solo se pone mas complicado :confounded:.

Ahora que ya tienes los archivos suplementarios como recordaras ya tienes los datos de expresion crudos de los samples del experimento en cuestion, sin embargo, como recordaras esto aun no es suficiente para integrar los datos pues tienen distinto identificadoress que nos los hacen comparables entre si:cry:.

Para solucionar este problema se tienen que hacer dos cosas: 
- En primera parsear los archivos suplementarios para poder recuperar: el valor de expresion, el probe_id, y la secuencia del probe
- Mapear las secuencias de los probes para saber a que locus tag pertenecen

# Parseo

Como viste en el entrenamiento a travez del uso de programacion basica en python o el uso de librerias como pandas es posible parsear facilemente archivos de diferentes formatos. Sin embargo, cuando no hay modulo de apoyo, para poder parserar los datos vas a tener que entender como es el formato de tu archivo para poder extraer lo que necesitas. 

Ahora bien, para hacer este vas a tener que buscar los manuales de los manufacturadores que explican el formato de sus archivos, sin embargo, para propositos de este ejercicio en teoria solo tienes que buscar el de uno, el de Agilent. Esto porque no te voy a perdir que parsees los archivos de Affymetrix y te voy a ayudar con los de Nimblegen :nerd_face:

In [12]:
# Empezamos con un pequeño ejemplo con solo un sample
import GEOparse as geo
import os
import ftplib
import gzip
import shutil

gsm = geo.get_GEO(geo="GSM6481068",destdir="Results")

# Ignora que tiene dos archivos suplementarios como es ejemplo pude revisarlos a mano xd
ftpath = gsm.metadata["supplementary_file"][0]

_,_,host,*path = ftpath.split("/")
path = os.path.join(*path)
ftp = ftplib.FTP(host)
ftp.login()

fileName = "GSM6481068_supp.gz"

with open(pathfile:=f"Results/{fileName}", "wb") as f:
  ftp.retrbinary(f"RETR {path}", f.write)

ftp.quit()

# Los archivos estan zipiados por lo que tienes que gunzipearlos

with gzip.open(pathfile, "rb") as gz:
            with open(GSM_pathunzip := pathfile.strip(".gz"), "wb") as out:
                shutil.copyfileobj(gz, out)

05-Aug-2025 20:21:54 DEBUG utils - Directory Results already exists. Skipping.
05-Aug-2025 20:21:54 INFO GEOparse - File already exist: using local version.
05-Aug-2025 20:21:54 INFO GEOparse - Parsing Results/GSM6481068.txt: 


In [13]:
# Podras pensar que solo con los samples basta, pero no en caso de nimblegen tambien se necesita el suplementario del GPL
gpl = geo.get_GEO(geo=(fileName := gsm.metadata["platform_id"][0]),destdir="Results")

ftpath = gpl.metadata["supplementary_file"][0]

_,_,host,*path = ftpath.split("/")
path = os.path.join(*path)
ftp = ftplib.FTP(host)
ftp.login()

fileName = f"{fileName}_supp.gz"

with open(pathfile:=f"Results/{fileName}", "wb") as f:
  ftp.retrbinary(f"RETR {path}", f.write)

ftp.quit()

with gzip.open(pathfile, "rb") as gz:
            # Recuerda guardar el path de los archivos de salida para mayor comodidad
            with open(GPL_pathunzip := pathfile.strip(".gz"), "wb") as out:
                shutil.copyfileobj(gz, out)

05-Aug-2025 20:22:07 DEBUG utils - Directory Results already exists. Skipping.
05-Aug-2025 20:22:07 INFO GEOparse - File already exist: using local version.
05-Aug-2025 20:22:07 INFO GEOparse - Parsing Results/GPL10416.txt: 
05-Aug-2025 20:22:07 DEBUG GEOparse - PLATFORM: GPL10416


In [14]:
import pandas as pd 
# Dado que son archivos tabulares es bastante simple el seleccionar lo que queremos
# GPL_pathunzip = "/export/storage/users/aggonzal/Induccion/Results/GPL10416_supp"
# GSM_pathunzip = "/export/storage/users/aggonzal/Induccion/Results/GSM6481068_supp"
gsm_supp = pd.read_csv(GSM_pathunzip,sep="\t",comment="#")
gpl_supp = pd.read_csv(GPL_pathunzip,sep="\t",comment="#")

In [15]:
# Veamos los archivos 
gsm_supp.head(2)

,IMAGE_ID,GENE_EXPR_OPTION,SEQ_ID,PROBE_ID,POSITION,X,Y,MATCH_INDEX,SEQ_URL,PM,MM
0,6610-423383-425-control_532,FORWARD,NC_000913,ECOLIP1,1,580,820,65084075,NaN,551.33,0.0
1,6610-423383-425-control_532,FORWARD,NC_000913,ECOLIP25,25,542,974,65084076,NaN,475.22,0.0


In [16]:
gpl_supp.head(2)

,PROBE_DESIGN_ID,CONTAINER,DESIGN_NOTE,SELECTION_CRITERIA,SEQ_ID,PROBE_SEQUENCE,MISMATCH,MATCH_INDEX,FEATURE_ID,ROW_NUM,COL_NUM,PROBE_CLASS,PROBE_ID,POSITION,DESIGN_ID,X,Y
0,4215_0001_0001,REVERSE,NaN,0,NC_000913,TGCCGGTAACGTCTGAGCGTACAATCCGGCGCGTTTTACCGCATTA...,0,65949753,65949753,1,1,experimental,ECOLIP1708766,1708766,4215,1,1
1,4215_0023_0001,FORWARD,NaN,0,NC_000913,GAAGCGGCTCATTAACAGGAGTATAATGATGGATTTTTCTTTAACT...,0,65158015,65158015,1,23,experimental,ECOLIP1775192,1775192,4215,23,1


Como puedes notar cada uno tiene informacion que nos interesa, por un lado el suplementario del GSM tiene los valores crudos de la columna 'PM', y en el suplementario del GPL la secuencia asociada a ello, pero hay algo que los une, y ese es el 'PROBE_ID'

In [17]:
# Entonces sabiendo eso podemos concatenar solo las columnas que nos interesan
merge_sups = gsm_supp.loc[:,["PROBE_ID","PM"]].merge(gpl_supp.loc[:,["PROBE_ID","PROBE_SEQUENCE"]],on="PROBE_ID",how="inner")
# on indica a través de qué columna va a buscar matches ---> Requiere que haya dos columnas homónimas en los dfs
# how que sólo se añadirán esas columnas que hagan match
# ---> inner para las coincidencias, left para todas las de la izquierda y coincidencias, right para viceversa, y outer para todas

Y listo tenemos nuestros datos de interes :smile:. Aunque bueno eso solo es para un tipo de archivo de nimblegen, los .pair, para el .xys cambian los nombre de las columnas pero eso te dejare descubrirlo :+1:

In [18]:
merge_sups

,PROBE_ID,PM,PROBE_SEQUENCE
0,ECOLIP1,551.33,AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATT...
1,ECOLIP1,551.33,TTTTAATCCACACAGAGACATATTGCCCGTTGCAGTCAGAATGAAA...
2,ECOLIP25,475.22,AGAAGCTGCTATCAGACACTCTTTTTTTAATCCACACAGAGACATA...
3,ECOLIP25,475.22,GCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGC...
4,ECOLIP49,412.11,TTACTCACGGCAGGTAACCAGTTCAGAAGCTGCTATCAGACACTCT...
...,...,...,...
775788,RANDOM00002887,134.22,CCTAAGGAATTCAAAACGCATGGCCGCTAGCGGAGCCGTGGCCATG...
775789,RANDOM00000700,127.44,GATGGATAACTCCAAATGTTGCCCAGTAAAGAACCGATCCCTGCAC...
775790,RANDOM00004985,137.44,GCGTGCGTGCAAAAGCGTTCATGGTCCAGCCCATGGGAGACCTACC...
775791,RANDOM00002723,135.67,TCATCTCGTATGCCGAGGCCGCGCCACACAACAGTTGCAGCTGACC...


# Mapeo

Ahora se viene lo que todos esperabamos, el mapeo de las secuencas, para ello como ya les platique previmente usamos el programa de bowtie2 para ello, aunque para eso necesitamos el genoma de referenci, aunque tu no tienes problema porque basta con que elijas el de escherichia 511145: https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/

Donde solo necesitas el fna y gff

Entonces ya con el genoma de referencia en tu directorio solo tienes que correr bowtie 2 de la siguiente manera

In [19]:
# Puedes correrlo en la linea de bash o en python como a mi me gusta
import os 
dir = "test"
preffix = "test"
if not os.path.exists(dir) or not os.path.isdir(dir): os.makedirs(dir)
# fnaFilePath = "/export/storage/users/aggonzal/Induccion/GCF_000005845.2_ASM584v2_genomic.fna.gz"
fnaFilePath = '/export/space3/users/majiso/Induccion/e_coli_genome_511145/GCF_000005845.2/GCF_000005845.2_ASM584v2_genomic.fna.gz'
# Como en muchos alineadores, necesitas indexar tu genoma para manejarlo de mejor manera
# Yo no lo corro porque ya lo tengo xd
command = f"/export/apps/bioconda/bin/bowtie2-build {fnaFilePath} {preffix}/{preffix} > {preffix + '.out'}" # Guarda output en el archivo test/test.out
os.system(command=command)

Building a SMALL index
Renaming test/test.3.bt2.tmp to test/test.3.bt2
Renaming test/test.4.bt2.tmp to test/test.4.bt2
Renaming test/test.1.bt2.tmp to test/test.1.bt2
Renaming test/test.2.bt2.tmp to test/test.2.bt2
Renaming test/test.rev.1.bt2.tmp to test/test.rev.1.bt2
Renaming test/test.rev.2.bt2.tmp to test/test.rev.2.bt2


0

In [20]:
merge_sups[["PROBE_ID","PROBE_SEQUENCE"]]

,PROBE_ID,PROBE_SEQUENCE
0,ECOLIP1,AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATT...
1,ECOLIP1,TTTTAATCCACACAGAGACATATTGCCCGTTGCAGTCAGAATGAAA...
2,ECOLIP25,AGAAGCTGCTATCAGACACTCTTTTTTTAATCCACACAGAGACATA...
3,ECOLIP25,GCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGC...
4,ECOLIP49,TTACTCACGGCAGGTAACCAGTTCAGAAGCTGCTATCAGACACTCT...
...,...,...
775788,RANDOM00002887,CCTAAGGAATTCAAAACGCATGGCCGCTAGCGGAGCCGTGGCCATG...
775789,RANDOM00000700,GATGGATAACTCCAAATGTTGCCCAGTAAAGAACCGATCCCTGCAC...
775790,RANDOM00004985,GCGTGCGTGCAAAAGCGTTCATGGTCCAGCCCATGGGAGACCTACC...
775791,RANDOM00002723,TCATCTCGTATGCCGAGGCCGCGCCACACAACAGTTGCAGCTGACC...


In [21]:
# Al alineador tienes que pasarle archivos fastamerge_sups[["PROBE_ID","PROBE_SEQUENCE"]]
with open(fasta_path:="Fasta_attemp_2.fasta","w") as file:
    for _,(probe,seq) in merge_sups[["PROBE_ID","PROBE_SEQUENCE"]].iterrows(): # Equivalente de items() en un dict ---> Regresa índice y tupla con las columnas
        file.write(f">{probe}\n{seq}\n")

In [22]:
# indice = "/export/storage/users/aggonzal/Induccion/test/test" # El indice generado con el genoma de referencia
indice = "/export/space3/users/majiso/Induccion/test/test"
out = "salida_2.sam" # Nombre del output de salida
stats = "salida_2.sam.out" # Archivo donde se guardan las estadisticas de corrida
command = f"/export/apps/bioconda/bin/bowtie2 -f {fasta_path} -x {indice} -S {out} 2> {stats}"
os.system(command=command)

0

In [23]:
# Para despues correr el featureCounts es mejor con archivos bam :3
command = f"/export/apps/bioconda/bin/samtools view -bS {out} | /export/apps/bioconda/bin/samtools sort -o {out.replace('.sam','.bam')}"
os.system(command=command)

0

In [24]:
import rpy2.robjects as robjects
import tempfile
import sys
from rpy2.rinterface_lib.embedded import RRuntimeError


# bamFile = "/export/storage/users/aggonzal/Induccion/salida_2.bam"
bamFile = "/export/storage/users/majiso/Induccion/salida_2.bam"

# Se supone que debe funcionar con zipeados pero mejor haz un gunzip
# gtf = "/export/storage/users/aggonzal/Induccion/GCF_000005845.2_ASM584v2_genomic.gtf"
gtf = "/export/space3/users/majiso/Induccion/e_coli_genome_511145/GCF_000005845.2/GCF_000005845.2_ASM584v2_genomic.gtf"

feature = "gene"
outDir = "feature_test_2"

command = f'''
        library(Rsubread)
        featureCounts("{bamFile}",
        annot.ext="{gtf}",
        isGTFAnnotationFile=T,
        nthreads=10,
        GTF.featureType="{feature}",
        reportReads="CORE",
        reportReadsPath="{outDir}", 
        verbose=F, tmpDir="{outDir}")
        '''
#robjects.r(command)
# Todo esto es solo para el manejo de errores con la libreria
with tempfile.TemporaryFile(mode="w") as tempFile:
        __stdout__ = sys.stdout
        sys.stdout = tempFile
        try:
                robjects.r(command)
        except RRuntimeError:
                print("Valio sorbete")
        
        sys.stdout = __stdout__



In [25]:
# Ya solo tienes que renotar los probes para que los valores esten asociados a sus locus tag correspondientes
import pandas as pd
features = pd.read_csv("feature_test_2/salida_2.bam.featureCounts",sep="\t",header=None)
features.columns = ["PROBE_ID","ASSIGN","VALUE","GENE"]
probe_feature = merge_sups.merge(features.loc[:,["PROBE_ID","GENE"]],on="PROBE_ID")

In [27]:
probe_feature.dropna()

,PROBE_ID,PM,PROBE_SEQUENCE,GENE
48,ECOLIP145,312.33,TTCATGGATGTTGTGTACTCTGTAATTTTTATCTGTCTGTGCGCTA...,b0001
49,ECOLIP145,312.33,TTCATGGATGTTGTGTACTCTGTAATTTTTATCTGTCTGTGCGCTA...,b0001
50,ECOLIP145,312.33,TTCATGGATGTTGTGTACTCTGTAATTTTTATCTGTCTGTGCGCTA...,b0001
51,ECOLIP145,312.33,TTCATGGATGTTGTGTACTCTGTAATTTTTATCTGTCTGTGCGCTA...,b0001
52,ECOLIP145,312.33,GGCATAGCGCACAGACAGATAAAAATTACAGAGTACACAACATCCA...,b0001
...,...,...,...,...
3091883,ECOLIP1579532,211.22,AATTGTATTGTCATACTGTCCGTTTTTCTTCAGTTGTTCGAGAATG...,b1498
3091884,ECOLIP1579532,211.22,AACGCATTCTCGAACAACTGAAGAAAAACGGACAGTATGACAATAC...,b1498
3091885,ECOLIP1579532,211.22,AACGCATTCTCGAACAACTGAAGAAAAACGGACAGTATGACAATAC...,b1498
3091886,ECOLIP1579532,211.22,AACGCATTCTCGAACAACTGAAGAAAAACGGACAGTATGACAATAC...,b1498


In [26]:
# Para la matriz solo necesitas el valor y el nombre del gen
matriz = probe_feature.dropna()[["GENE","PM"]].drop_duplicates()
matriz.set_index("GENE",inplace=True)
# En este caso sabemos que todos nuestros valor pertenecen a un solo sample
matriz.rename(columns={"PM":"GSM6481068"},inplace=True)

In [27]:
# Y listo tenemos una columna de la matriz de expresion
matriz.head()

,GSM6481068
GENE,
b0001,312.33
b0001,473.33
b0001,739.33
b0001,454.78
b0001,162.22


# Actividad :horse: :fire: :fire:

Espero que los codigos que te di se pudieran entender :smiley_cat:, pero ahora es tiempo de ponerlo en practica como ya sabes, pero tranquilo si alguna cosa no funciona lo podemos revisar.

Para este modulo solo te voy a pedir lo siguiente:
Para aquellos gsms a los que les descargaste suplementarios, exclusivamente aquellos con manufacturadores de Nimblegen o Agilent; deberas realizar un parseo de los archivos suplementarios para la extraccion de minimo el valor crudo, la secuencia y el probe_id. 

Para posteriormente mapear las secuencias y recuperar el locus tag asociado a cada valor, ojo si algunas de las secuencias no mapean y se pierden esta bien. 

Finalmente una vez recuperados los locus tag construye una matriz de expresion donde las filas sean los locus tag y las columnas los gsms, para todos los gsms que pudiste recuperar.

## 1. Recuperar el DataFrame
> Ya existe como tsv, sólo es bajarlo de nuevo

In [28]:
import pandas as pd
import GEOparse as geo
import numpy as np
import ftplib
import gzip
import ast
import re
import os
from shutil import copyfileobj

In [29]:
def hacer_eval(celda):
    try:
        return ast.literal_eval(celda)
        df = df.apply(lambda col: col.map(hacer_eval)) 
    except (ValueError, SyntaxError):
        return celda

In [30]:
data_gsm = pd.read_csv("Results/e_coli_k12_microarrays_withpaths.tsv",sep="\t",index_col=0)
data_gsm = data_gsm.apply(lambda col: col.map(hacer_eval)) 

In [32]:
data_gpl = pd.read_csv("Results/gpl_metadata.tsv",sep="\t",index_col=0)
data_gpl = data_gpl.apply(lambda col: col.map(hacer_eval)) 

In [33]:
filtered_data = data_gsm[(data_gsm.channel_count.astype(str).str.contains("1")) & (data_gsm.manufacturer.isin(["Agilent","Nimblegen"])) & (data_gsm.suplementary_files_number == 1) & (data_gsm.path.notna()) & (data_gsm.taxid_ch1.astype(str).str.contains("511145"))]

In [34]:
filtered_data

,title,geo_accession,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,taxid_ch1,...,contact_web_link,biomaterial_provider_ch1,biomaterial_provider_ch2,technology,valid_ch1_2,has_suplementary_file,suplementary_files_number,path,manufacturer,extension
GSM8125782,[FounderA_Round0_rep1],[GSM8125782],[Public on Nov 24 2024],[Mar 05 2024],[Nov 25 2024],[RNA],[1],"[Founder A of MA experiment, biological replic...",[Escherichia coli str. K-12 substr. MG1655],[511145],...,NaN,NaN,NaN,True,True,True,1.0,geo/samples/GSM8125nnn/GSM8125782/suppl/GSM812...,Agilent,txt
GSM8125783,[FounderA_Round0_rep2],[GSM8125783],[Public on Nov 24 2024],[Mar 05 2024],[Nov 25 2024],[RNA],[1],"[Founder A of MA experiment, biological replic...",[Escherichia coli str. K-12 substr. MG1655],[511145],...,NaN,NaN,NaN,True,True,True,1.0,geo/samples/GSM8125nnn/GSM8125783/suppl/GSM812...,Agilent,txt
GSM8125784,[FounderB_Round0_rep1],[GSM8125784],[Public on Nov 24 2024],[Mar 05 2024],[Nov 25 2024],[RNA],[1],"[Founder B of MA experiment, biological replic...",[Escherichia coli str. K-12 substr. MG1655],[511145],...,NaN,NaN,NaN,True,True,True,1.0,geo/samples/GSM8125nnn/GSM8125784/suppl/GSM812...,Agilent,txt
GSM8125785,[FounderB_Round0_rep2],[GSM8125785],[Public on Nov 24 2024],[Mar 05 2024],[Nov 25 2024],[RNA],[1],"[Founder B of MA experiment, biological replic...",[Escherichia coli str. K-12 substr. MG1655],[511145],...,NaN,NaN,NaN,True,True,True,1.0,geo/samples/GSM8125nnn/GSM8125785/suppl/GSM812...,Agilent,txt
GSM8125786,[DescendantOfFounderA_H03_Round10],[GSM8125786],[Public on Nov 24 2024],[Mar 05 2024],[Nov 25 2024],[RNA],[1],"[Isolate of MA experiment, Round 10, Descendan...",[Escherichia coli str. K-12 substr. MG1655],[511145],...,NaN,NaN,NaN,True,True,True,1.0,geo/samples/GSM8125nnn/GSM8125786/suppl/GSM812...,Agilent,txt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM509042,[coli_lactose_timepoint2_rep3],[GSM509042],[Public on Feb 13 2010],[Feb 12 2010],[May 20 2011],[RNA],[1],[Escherichia coli_stress experiment],[Escherichia coli str. K-12 substr. MG1655],[511145],...,NaN,NaN,NaN,True,True,True,1.0,geo/samples/GSM509nnn/GSM509042/suppl/GSM50904...,Agilent,txt
GSM509043,[coli_lactose_timepoint3_rep3],[GSM509043],[Public on Feb 13 2010],[Feb 12 2010],[May 20 2011],[RNA],[1],[Escherichia coli_stress experiment],[Escherichia coli str. K-12 substr. MG1655],[511145],...,NaN,NaN,NaN,True,True,True,1.0,geo/samples/GSM509nnn/GSM509043/suppl/GSM50904...,Agilent,txt
GSM509044,[coli_lactose_timepoint4_rep3],[GSM509044],[Public on Feb 13 2010],[Feb 12 2010],[May 20 2011],[RNA],[1],[Escherichia coli_stress experiment],[Escherichia coli str. K-12 substr. MG1655],[511145],...,NaN,NaN,NaN,True,True,True,1.0,geo/samples/GSM509nnn/GSM509044/suppl/GSM50904...,Agilent,txt
GSM509045,[coli_lactose_timepoint5_rep3],[GSM509045],[Public on Feb 13 2010],[Feb 12 2010],[May 20 2011],[RNA],[1],[Escherichia coli_stress experiment],[Escherichia coli str. K-12 substr. MG1655],[511145],...,NaN,NaN,NaN,True,True,True,1.0,geo/samples/GSM509nnn/GSM509045/suppl/GSM50904...,Agilent,txt


#### Descarga de los suplementarios de las GPLs

In [56]:
filtered_data.extension.drop_duplicates() # No encontré archivos .xys

GSM8125782     txt
GSM4285723    pair
Name: extension, dtype: object

In [36]:
data_gpl.supplementary_file

GPL32387    [ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL3...
GPL35385                                                  NaN
GPL18948    [ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL1...
GPL10416    [ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL1...
GPL13359                                                  NaN
                                  ...                        
GPL6427                                                   NaN
GPL6540                                                   NaN
GPL6570                                                   NaN
GPL73                                                     NaN
GPL189                                                    NaN
Name: supplementary_file, Length: 72, dtype: object

In [37]:
for gsm, row in filtered_data.iterrows():
    # data_gpl.loc[(gpl := row.platform_id[0]), 'supplementary_file'][0]
    print(row.platform_id[0])

GPL18948
GPL18948
GPL18948
GPL18948
GPL18948
GPL18948
GPL18948
GPL18948
GPL18948
GPL18948
GPL18948
GPL18948
GPL18948
GPL18948
GPL18948
GPL18948
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL25406
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
GPL14649
G

In [38]:
for gsm, row in filtered_data.iterrows():
    lista = data_gpl.loc[(gpl := row.platform_id[0]), 'supplementary_file']
    ftpath = lista[0] if isinstance(lista, list) else None
    print(ftpath)
    

ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL18nnn/GPL18948/suppl/GPL18948_platform_design_file1.txt.gz
ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL18nnn/GPL18948/suppl/GPL18948_platform_design_file1.txt.gz
ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL18nnn/GPL18948/suppl/GPL18948_platform_design_file1.txt.gz
ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL18nnn/GPL18948/suppl/GPL18948_platform_design_file1.txt.gz
ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL18nnn/GPL18948/suppl/GPL18948_platform_design_file1.txt.gz
ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL18nnn/GPL18948/suppl/GPL18948_platform_design_file1.txt.gz
ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL18nnn/GPL18948/suppl/GPL18948_platform_design_file1.txt.gz
ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL18nnn/GPL18948/suppl/GPL18948_platform_design_file1.txt.gz
ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL18nnn/GPL18948/suppl/GPL18948_platform_design_file1.txt.gz
ftp://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL18nnn/GPL18948/suppl/GPL18948

In [39]:
def listar() -> list:
    
    if not os.path.exists('Results/GPL_Supplementaries') or not os.path.isdir('Results/GPL_Supplementaries'): # Sólo validación
        os.makedirs('Results/GPL_Supplementaries')

    return [file for file in os.listdir('Results/GPL_Supplementaries') if '_suplementary_file.' in file]

In [59]:
def find_data_start_line(path) -> int:
    with open(path) as file:
        for i, line in enumerate(file):
            if 'FeatureNum' in line or 'IMAGE_ID' in line: # Dependiendo del archivo (Columnas que hay en .txt y .pair)
                return i
    return 0 # Si no lo encuentra

def read_gsm_supp(path) -> pd.DataFrame: # Hice un Head de los supps porque no corría, y vi que en el caso de los txt, la info no comienza hasta líneas abajo # A partir de qué línea empieza el DataFrame real
    return pd.read_csv(path, sep='\t', comment='#', skiprows=find_data_start_line(path)) # Intenté usar aquí el walrus operator pero me decía que la asignación interna no estaba definida...


In [40]:
def get_supp_path(row, data_gpl, gpl_supps_previos) -> tuple[str, str, str]:
    ftpath = data_gpl.loc[(gpl := row.platform_id[0]), 'supplementary_file'][0]
    _,_,host,*path = ftpath.split('/')
    path = os.path.join(*path)
    fileName = f'{gpl}_suplementary_file.{path.split(".")[-2]}'

    return (host, gpl, path, fileName) if not fileName in gpl_supps_previos else (None, gpl, None, fileName)

In [41]:
def download_gpl_supps(zipped, fileName, path, errores) -> bool:
    try: 
        with open(zipped, 'wb') as f: # Modo de escritura binario ---> Sólo bytes, no strings (b)
            ftp.retrbinary(f'RETR {path}', f.write)

    except Exception as e:
        errores['no descargados'].append((fileName, str(e)))
        return False
    
    return True

In [42]:
def unzip_gpl_supps(zipped, unzipped, fileName, errores) -> bool:
    try:
        with gzip.open(zipped) as gz:
            with open(unzipped, 'wb') as out: # Quita el .gz
                copyfileobj(gz, out) # Desde un fsrc (source) hasta un fdst (destination)

    except Exception as e:
        errores['no descomprimidos'].append((fileName, str(e)))
        return False
    
    return True

In [43]:
def merge_dfs(fileName, gsm_supp) -> pd.DataFrame:
    gpl_supp = pd.read_csv(f'Results/GPL_Supplementaries/{fileName}', sep="\t", comment='#')
    return gsm_supp.merge(gpl_supp.loc[:,['PROBE_ID', 'PROBE_SEQUENCE']], on='PROBE_ID', how='left') # Conserva DF original pero añade la columna

In [44]:
def homogenize(gsm_supp) -> None:
    gsm_supp.rename(columns={
            'ProbeName': 'PROBE_ID',
            'Sequence': 'PROBE_SEQUENCE',
            'gProcessedSignal': 'PM'
        }, inplace=True)

In [46]:
def generate_sam(gsm, fasta_path, indice_e_coli) -> None:
    out = f'{gsm}.sam' # Output
    stats = f'{gsm}.sam.out' # Stats
    command = f'/export/apps/bioconda/bin/bowtie2 -f {fasta_path} -x {indice_e_coli} -S {out} 2> {stats}'
    os.system(command=command)

def count_features(out) -> None:
    command = f'/export/apps/bioconda/bin/samtools view -bS {out} | /export/apps/bioconda/bin/samtools sort -o {out.replace(".sam",".bam")}'
    os.system(command=command)

In [47]:
def r_unzip(bamFile, gtf, feature, outDir) -> None:
    command = f'''
            library(Rsubread)
            featureCounts("{bamFile}",
            annot.ext="{gtf}",
            isGTFAnnotationFile=T,
            nthreads=10,
            GTF.featureType="{feature}",
            reportReads="CORE",
            reportReadsPath="{outDir}", 
            verbose=F, tmpDir="{outDir}")
            '''
    # Manejo de errores, en lugar de sólo 'robjects.r(command)'
    with tempfile.TemporaryFile(mode='w') as tempFile:
        __stdout__ = sys.stdout
        sys.stdout = tempFile
        try:
            robjects.r(command)
        except RRuntimeError:
            print('Ya mamó :,)')
            
        sys.stdout = __stdout__

In [ ]:
gpl_supps_previos = listar()

In [68]:
# Prueba

errores = {key: [] for key in ['no descargados', 'no descomprimidos']} # Listas vacías, independientes

ftpath = data_gpl.loc[(gpl := filtered_data.loc['GSM8125782'].platform_id[0]), 'supplementary_file'][0]
_,_,host,*path = ftpath.split('/')
path = os.path.join(*path)
fileName = f'{gpl}_suplementary_file.{path.split(".")[-2]}'

ftp = ftplib.FTP(host)
ftp.login()

zipped = (unzipped := f'Results/GPL_Supplementaries/{fileName}') + '.gz'

with open(zipped, 'wb') as f: # Modo de escritura binario ---> Sólo bytes, no strings (b)
    ftp.retrbinary(f'RETR {path}', f.write)

unzip_gpl_supps(zipped, unzipped, fileName, errores)
ftp.quit()

'221 Goodbye.'

In [86]:
prueba = pd.read_csv('/export/space3/users/majiso/Induccion/Results/GSM_Supplementaries/GSM8125782_suplementary_file.txt', sep='\t', header=1, comment='#', on_bad_lines='skip', engine='python', skiprows=8)
prueba.columns

Index(['FEATURES', 'FeatureNum', 'Row', 'Col', 'SubTypeMask', 'ControlType',
       'ProbeName', 'SystematicName', 'PositionX', 'PositionY',
       'gProcessedSignal', 'gProcessedSigError', 'gMedianSignal',
       'gBGMedianSignal', 'gBGPixSDev', 'gIsSaturated', 'gIsFeatNonUnifOL',
       'gIsBGNonUnifOL', 'gIsFeatPopnOL', 'gIsBGPopnOL', 'IsManualFlag',
       'gBGSubSignal', 'gIsPosAndSignif', 'gIsWellAboveBG', 'SpotExtentX',
       'gBGMeanSignal'],
      dtype='object')

In [78]:
prueba = pd.read_csv('/export/space3/users/majiso/Induccion/Results/GPL_Supplementaries/GPL18948_suplementary_file.txt', sep='\t', header=0, comment='#', on_bad_lines='skip', engine='python')
prueba

,Column,Row,ProbeName,ID,RefNumber,ControlType,GeneName,TopHit,Description,Go,ChromosomalLocation,EntrezGeneID
0,1,1,GE_BrightCorner,GE_BrightCorner,1,pos,GE_BrightCorner,NaN,Unknown,NaN,Unknown,NaN
1,2,1,DarkCorner,DarkCorner,2,pos,DarkCorner,NaN,Unknown,NaN,Unknown,NaN
2,3,1,DarkCorner,DarkCorner,3,pos,DarkCorner,NaN,Unknown,NaN,Unknown,NaN
3,4,1,NaN,31181,4,FALSE,NaN,NaN,Unknown,NaN,unmapped,NaN
4,5,1,NaN,55868,5,FALSE,NaN,NaN,Unknown,NaN,unmapped,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
31633,99820q30557405i4634399879999977876,694t31S4586506870t9374595y586875378344451y0908...,9679839,None,None,None,None,None,None,None,None,None
31634,3005024199S69515703k59999,9679839aem629029384D57|87820056781055706103549...,4v21y0m81055G31290983980p29k36Qa67945098 507o7...,38682e70718619959b256d694t31S45865068706E91rd2...,None,None,None,None,None,None,None,None
31635,964149168990|371716b 9s706l31396977306699dt90b...,None,None,None,None,None,None,None,None,None,None,None
31636,769068258em336225d806909i2388y4i979912100,None,None,None,None,None,None,None,None,None,None,None


In [67]:
print(filtered_data.loc['GSM8125782'].platform_id[0])

GPL18948


In [62]:
summaries = {}
contador = 0
indice = 0
errores = {key: [] for key in ['no descargados', 'no descomprimidos']} # Listas vacías, independientes
n_archivos = filtered_data[filtered_data.manufacturer == 'Nimblegen'].shape[0] # 216
sample_info = {}
matrixes = []

# Mapeo ---> Mismo para todas las samples
dir = 'ecoli_index'
preffix = 'ecoli_index'
if not os.path.exists(dir) or not os.path.isdir(dir): os.makedirs(dir)
fnaFilePath = '/export/space3/users/majiso/Induccion/e_coli_genome_511145/GCF_000005845.2/GCF_000005845.2_ASM584v2_genomic.fna.gz' # Genoma de referencia
command = f"/export/apps/bioconda/bin/bowtie2-build {fnaFilePath} {preffix}/{preffix} > {preffix + '.out'}" # Guarda output en el archivo test/test.out
os.system(command=command)
indice_e_coli = '/export/space3/users/majiso/Induccion/ecoli_index/ecoli_index'

gtf = '/export/space3/users/majiso/Induccion/e_coli_genome_511145/GCF_000005845.2/GCF_000005845.2_ASM584v2_genomic.gtf'
feature = 'gene'
outDir = 'features_ecoli'

# Para cada GSM
for gsm, row in filtered_data.iterrows():
    indice += 1

    # Saber qué descargar
    gsm_supp_path = f'Results/GSM_Supplementaries/{f"{gsm}_suplementary_file.{row.extension}"}' # A nivel GSM

    # Para fabricante Nimblegen
    if row.manufacturer == 'Nimblegen': # Caso contrario sería Agilent, que no necesita datos de GPL
        gsm_supp = read_gsm_supp(gsm_supp_path) # Para que haga el skip

        host, gpl, path, fileName = get_supp_path(row, data_gpl, gpl_supps_previos)

        # Si no se tenía ya descargado
        if path:
            # Apertura de conexión
            ftp = ftplib.FTP(host)
            ftp.login()

            zipped = (unzipped := f'Results/GPL_Supplementaries/{fileName}') + '.gz'

            # Descarga
            if not download_gpl_supps(zipped, fileName, path, errores):
                ftp.quit()
                continue

            # Descompresión
            if not unzip_gpl_supps(zipped, unzipped, fileName, errores):
                ftp.quit()
                continue

            # Cierre de conexión FTP
            ftp.quit()

            contador += 1
        print(f'Progreso:\t{round((((indice) / n_archivos) * 100), 5)}%')

        # Obtención y merge de DataFrames
        gsm_supp = merge_dfs(fileName, gsm_supp)

    # En este punto, sea del fabricante que sea, ambos tienen la información requerida en 'gsm_supp', sólo falta homogenizar los nombres de las columnas

    # Cuando el fabricante sea Agilent
    else:
        gsm_supp = read_gsm_supp(gsm_supp_path)
        homogenize(gsm_supp) # Según encontré en el manual de Agilent que venían los campos

    # Validación de campos
    required_cols = ['PROBE_ID', 'PROBE_SEQUENCE']
    missing_cols = [col for col in required_cols if col not in gsm_supp.columns]

    if missing_cols:
        print(f"{gsm}: columnas faltantes: {missing_cols}. Saltando...")
        continue

    # Archivos fasta
    with open(fasta_path := f'{gsm}_merged_probes.fasta', 'w') as file:
        file.writelines(f'>{probe}\n{seq}\n' for _, (probe, seq) in gsm_supp[['PROBE_ID', 'PROBE_SEQUENCE']].iterrows())

    # Archivos SAM
    generate_sam(gsm, fasta_path, indice_e_coli)

    # Features Count
    count_features(out)

    bamFile = f'/export/storage/users/majiso/Induccion/{gsm}.bam'

    # Descompresión, por si acaso
    r_unzip(bamFile, gtf, feature, outDir)
    
    # Asociación valor-locus tag
    features = pd.read_csv(f'features_ecoli/{gsm}.bam.featureCounts', sep='\t', header=None)
    features.columns = ['PROBE_ID', 'ASSIGN', 'VALUE', 'GENE']
    sample_info[gsm] = (probe_feature := gsm_supp[['PROBE_ID', 'PM', 'PROBE_SEQUENCE']].merge(features[['PROBE_ID', 'GENE', 'ASSIGN', 'VALUE']], on='PROBE_ID', how='left')) # Dict de la info

    # Exportación de dataframe
    probe_feature.to_csv(f'Results/Dataframes/{gsm}.tsv', sep='\t')
                        
    # Matriz
    matrix = probe_feature.dropna()[['GENE', 'PM']].drop_duplicates()
    matrix.set_index('GENE', inplace=True)
    matrix.rename(columns={'PM': gsm}, inplace=True)
    matrixes.append(matrix)

# Unión de matrices en una sola por GENEs
matriz_final = pd.concat(matrixes, axis=1) # Pega horizontalmente los DFs de las matrices de cada gen almacenados
matriz_final.to_csv('Results/matrixes/Global_Ecoli_matrix.tsv', sep='\t')


print(f'{contador}/{n_archivos} archivos nuevos descargados (sin contar .gz)')
print(f'Archivos en el directorio Results/GPL_Supplementaries/ {os.listdir("Results/GPL_Supplementaries/")}')
print(f"CSV's en el directorio Results/Dataframes/ {os.listdir('Results/Dataframes/')}")
print(f'Matriz generada con información de {len(matrixes)} GSMs')

Building a SMALL index
Renaming ecoli_index/ecoli_index.3.bt2.tmp to ecoli_index/ecoli_index.3.bt2
Renaming ecoli_index/ecoli_index.4.bt2.tmp to ecoli_index/ecoli_index.4.bt2
Renaming ecoli_index/ecoli_index.1.bt2.tmp to ecoli_index/ecoli_index.1.bt2
Renaming ecoli_index/ecoli_index.2.bt2.tmp to ecoli_index/ecoli_index.2.bt2
Renaming ecoli_index/ecoli_index.rev.1.bt2.tmp to ecoli_index/ecoli_index.rev.1.bt2
Renaming ecoli_index/ecoli_index.rev.2.bt2.tmp to ecoli_index/ecoli_index.rev.2.bt2


GSM8125782: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125783: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125784: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125785: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125786: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125787: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125788: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125789: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125790: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125791: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125792: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125793: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125794: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125795: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125796: columnas faltantes: ['PROBE_SEQUENCE']. Saltando...
GSM8125797: columnas faltantes: ['PROBE_

EOFError: 